# Task 2 classifier on Kaggle
Select a GPU accelerator and enable Internet. No specialist checkpoint is needed: this classifier starts fresh. Run cells in order. Output is saved in Kaggle working storage, NOT Google Drive. Download the backup before ending the session. This is a ten-epoch provisional baseline; classifier Optuna remains pending.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
import shutil
output = Path('/kaggle/working/task2-classifier')
# FIRST RUN: leave as None. Later sessions: attach the extracted classifier backup
# as a private dataset and enter the path to its last.pt here.
resume_input = None
if resume_input is not None and not output.exists():
    checkpoint = Path(resume_input)
    assert checkpoint.is_file(), 'Set the exact attached classifier last.pt path'
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    assert 'best' in saved['progress'] and 'hashes' in saved['data_signature'], 'Not a classifier checkpoint'
    assert not saved['data_signature']['train_limit'] and not saved['data_signature']['val_images'], 'Debug checkpoint'
    shutil.copytree(checkpoint.parent, output)


In [ ]:
command = [sys.executable, '-m', 'training.train_classifier',
    '--device', 'cuda', '--max-hours', '2', '--output-dir', str(output)]
if (output / 'last.pt').exists():
    command += ['--resume', str(output / 'last.pt')]
subprocess.run(command, cwd=repo, check=True)


In [ ]:
import json
from IPython.display import display, FileLink
history = json.loads((output / 'history.json').read_text())
print('Completed epochs:', history[-1]['epoch'] if history else 0)
if (output / 'best_validation.json').exists():
    report = json.loads((output / 'best_validation.json').read_text())
    print('Best epoch:', report['epoch'])
    for metric in ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']:
        print(metric, report[metric])
    print('Class order:', report['class_order'])
    print('Confusion matrix: true rows, predicted columns')
    for row in report['normalized_confusion']: print(row)
else:
    print('No completed validation yet; resume from last.pt')
backup = shutil.make_archive('/kaggle/working/task2_classifier_backup', 'zip', output.parent, output.name)
display(FileLink(backup))


Download task2_classifier_backup.zip using the link or Kaggle output file browser. Preserve the whole ZIP locally and on Drive. Send history.json and best_validation.json for review. If completed epochs is below10, rerun training in this session or attach the newest extracted backup in a later session and set resume_input. Do not restart from scratch. The two-hour budget is a soft session limit, not a guarantee. Each batch is class balanced; validation has unequal class supports, so inspect macro F1 and per-class recall as well as accuracy. Official test data remains unused.
